# Ответ рецензенту 1: дополнительные проверки и данные для воспроизведения
**Статья:** «Надёжность оценки моделей машинного обучения на малых несбалансированных онкологических данных»

Блокнот выполняет:
1. подсчёт идентичных записей в наборе Cervical Cancer на каждом этапе подготовки данных;
2. анализ чувствительности без признаков ранее установленных диагнозов (Dx);
3. сохранение очищенных наборов данных, индексов всех разбиений и проверочных прогнозов.

Запуск: «Среда выполнения → Выполнить все» (≈ 10–15 минут).

In [ ]:
!pip install kagglehub -q
import io, os, glob, zipfile, requests, warnings
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             recall_score, balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")
SEED = 42
MODELS = ["LR", "RF", "GB", "SVM"]
from sklearn.model_selection import train_test_split
os.makedirs("repro", exist_ok=True)

In [ ]:
# UCI HCC Survival (165 пациентов)
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/423/hcc+survival.zip", timeout=60).content))
name = [n for n in z.namelist() if n.endswith("hcc-data.txt") and "MACOSX" not in n][0]
hcc = pd.read_csv(z.open(name), header=None, na_values="?")
hcc.columns = ["Gender","Symptoms","Alcohol","HBsAg","HBeAg","HBcAb","HCVAb","Cirrhosis","Endemic",
 "Smoking","Diabetes","Obesity","Hemochro","AHT","CRI","HIV","NASH","Varices","Spleno","PHT","PVT",
 "Metastasis","Hallmark","Age","Grams_day","Packs_year","PS","Encephalopathy","Ascites","INR","AFP",
 "Hemoglobin","MCV","Leucocytes","Platelets","Albumin","Total_Bil","ALT","AST","GGT","ALP","TP",
 "Creatinine","Nodule","Major_Dim","Dir_Bil","Iron","Sat","Ferritin","Class"]
y_hcc = (hcc["Class"] == 0).astype(int); X_hcc = hcc.drop(columns=["Class"])

# Cervical Cancer Risk Factors (858 пациенток), целевая переменная – биопсия
z = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://archive.ics.uci.edu/static/public/383/cervical+cancer+risk+factors.zip", timeout=60).content))
cerv = pd.read_csv(z.open("risk_factors_cervical_cancer.csv"), na_values="?")
y_cerv = cerv["Biopsy"].astype(int)
X_cerv = cerv.drop(columns=["Hinselmann","Schiller","Citology","Biopsy"])

# Kaggle Survey Lung Cancer (309 записей)
import kagglehub
path = kagglehub.dataset_download("mysarahmadbhat/lung-cancer")
lung = pd.read_csv(glob.glob(os.path.join(path, "*.csv"))[0]); lung.columns = [c.strip() for c in lung.columns]
y_lung = lung["LUNG_CANCER"].map({"YES": 1, "NO": 0})
X_lung = lung.drop(columns=["LUNG_CANCER"]).copy()
X_lung["GENDER"] = X_lung["GENDER"].map({"M": 1, "F": 0})
for c in X_lung.columns:
    if c not in ["AGE", "GENDER"]: X_lung[c] = X_lung[c] - 1

def clean(X, y, thr=0.40):
    """Очистка без использования целевой переменной (как в статье)."""
    X = X.copy()
    X = X.drop(columns=X.columns[X.isna().mean() > thr])
    X = X.drop(columns=[c for c in X.columns if X[c].nunique(dropna=True) <= 1])
    full = X.assign(__y=y.values).drop_duplicates()
    y2 = full.pop("__y").astype(int).reset_index(drop=True)
    return full.reset_index(drop=True), y2

RAW = {"HCC": (X_hcc, y_hcc), "Cervical": (X_cerv, y_cerv), "Lung": (X_lung, y_lung)}
DATA = {k: clean(*v) for k, v in RAW.items()}
for k, (X, y) in DATA.items():
    print(f"{k:9s} n={len(y):4d}  признаков={X.shape[1]:2d}  положительных={y.sum():3d} ({y.mean()*100:.1f}%)")

In [ ]:
def make_model(name, seed=SEED):
    return {"LR":  LogisticRegression(max_iter=2000, C=1.0),
            "RF":  RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=1),
            "GB":  GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=seed),
            "SVM": SVC(kernel="rbf", probability=True, random_state=seed)}[name]

def pipe(name):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("clf", make_model(name))])

## 1. Идентичные записи в наборе Cervical Cancer на разных этапах

In [ ]:
stages = {}
stages["1. Исходный файл (все 36 столбцов)"] = cerv.duplicated().sum()
c2 = cerv.drop(columns=["Hinselmann", "Schiller", "Citology"])
stages["2. После исключения трёх диагностических тестов (32 признака + Biopsy)"] = c2.duplicated().sum()
Xc_tmp = X_cerv.copy()
Xc_tmp = Xc_tmp.drop(columns=Xc_tmp.columns[Xc_tmp.isna().mean() > 0.40])
Xc_tmp = Xc_tmp.drop(columns=[c for c in Xc_tmp.columns if Xc_tmp[c].nunique(dropna=True) <= 1])
c3 = Xc_tmp.assign(Biopsy=y_cerv.values)
stages["3. После удаления 2 признаков с пропусками > 40% и 2 константных (28 признаков + Biopsy)"] = c3.duplicated().sum()
for k, v in stages.items():
    print(f"{k}: {v}")
# одинаковые признаки, но разные метки
feat_dups = Xc_tmp.duplicated(keep=False)
grp = c3[feat_dups].groupby(list(Xc_tmp.columns), dropna=False)["Biopsy"].nunique()
print("Групп с совпадающими признаками, но разной меткой:", int((grp > 1).sum()))
print("Итоговый объём набора после удаления идентичных записей:", len(DATA["Cervical"][1]))
pd.Series(stages).to_csv("repro/cervical_duplicates_by_stage.csv")

## 2. Анализ чувствительности без признаков ранее установленных диагнозов (Dx)
Признаки `Dx:Cancer`, `Dx:CIN`, `Dx:HPV`, `Dx` описывают ранее установленные диагнозы. Чтобы исключить возможность того, что они недоступны на момент оценки риска, модели оцениваются с этими признаками и без них на одних и тех же записях и разбиениях (CV 5×5).

In [ ]:
Xc, yc = DATA["Cervical"]
dx_cols = [c for c in Xc.columns if c.startswith("Dx")]
print("Исключаемые признаки:", dx_cols)
Xc_nodx = Xc.drop(columns=dx_cols)
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2026).split(Xc, yc))

def auc_pr(X, y, tr, te, m):
    p = pipe(m).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y.iloc[te], p), average_precision_score(y.iloc[te], p)

rows = []
for label, X in [("с Dx", Xc), ("без Dx", Xc_nodx)]:
    res = Parallel(n_jobs=-1)(delayed(auc_pr)(X, yc, tr, te, m) for tr, te in splits for m in MODELS)
    k = 0
    for f, (tr, te) in enumerate(splits):
        for m in MODELS:
            rows.append({"variant": label, "fold": f, "model": m, "AUC_ROC": res[k][0], "PR_AUC": res[k][1]}); k += 1
dx = pd.DataFrame(rows); dx.to_csv("repro/cervical_dx_sensitivity.csv", index=False)
print(dx.groupby(["model", "variant"])[["AUC_ROC", "PR_AUC"]].agg(["mean", "std"]).round(3).to_string())

## 3. Данные для воспроизведения
Сохраняются очищенные наборы данных, индексы тестовых частей всех разбиений, использованных в статье, и прогнозы, по которым рассчитывались показатели калибровки.

In [ ]:
# 3.1 Очищенные наборы данных (номер строки = row_id в файлах индексов)
for ds, (X, y) in DATA.items():
    X.assign(target=y.values).to_csv(f"repro/clean_{ds}.csv", index_label="row_id")

# 3.2 Эксперимент 1: 200 однократных разбиений 70/30 (random_state = 0…199)
rows = []
for ds, (X, y) in DATA.items():
    for i in range(200):
        _, te = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=i)
        rows += [(ds, i, int(r)) for r in te]
pd.DataFrame(rows, columns=["dataset", "split_seed", "test_row_id"]).to_csv("repro/splits_exp1_single_70_30.csv", index=False)

# 3.3 Эксперимент 1б/2: CV 5×5, 20 повторов (random_state = 1000…1019)
rows = []
for ds, (X, y) in DATA.items():
    for s in range(20):
        for k, (tr, te) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=1000 + s).split(X, y)):
            rows += [(ds, s, k // 5, k % 5, int(r)) for r in te]
pd.DataFrame(rows, columns=["dataset", "seed", "repeat", "fold", "test_row_id"]).to_csv("repro/splits_exp1b_cv5x5.csv", index=False)

# 3.4 Эксперимент 4: прогнозы на тестовых фолдах (random_state = 7000), по ним построены калибровочные кривые
rows = []
for ds, (X, y) in DATA.items():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7000)
    fold_of = np.empty(len(y), dtype=int)
    for k, (_, te) in enumerate(cv.split(X, y)): fold_of[te] = k
    for m in MODELS:
        p = cross_val_predict(pipe(m), X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        rows += [(ds, m, i, int(fold_of[i]), int(y.iloc[i]), round(float(p[i]), 6)) for i in range(len(y))]
pd.DataFrame(rows, columns=["dataset", "model", "row_id", "fold", "y_true", "p_pred"]).to_csv("repro/oof_predictions_exp4.csv", index=False)
print("Сохранено:", sorted(os.listdir("repro")))

In [ ]:
from google.colab import files
with zipfile.ZipFile("reviewer1_repro.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir("repro"): zf.write(os.path.join("repro", f))
files.download("reviewer1_repro.zip")